In [1]:
!pip install -q flwr["simulation"]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.4/71.4 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 106.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 122.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.7/251.7 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 752.9/752.9 kB 44.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
black 26.1.0 requires pathspec>=1.0.0, but you have pathspec 0.12.1 which is incompa

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import flwr as fl
from collections import OrderedDict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
image_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),   # Unified input size
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],   # ImageNet mean
            std=[0.229, 0.224, 0.225]     # ImageNet std
        )
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
}

data_dir = "/kaggle/input/datasets/rishwanthj/xray-ds/split_dataset"

train_dataset = datasets.ImageFolder(root=f"{data_dir}/train", transform=image_transforms['train'])
val_dataset = datasets.ImageFolder(root=f"{data_dir}/val", transform=image_transforms['val'])
test_dataset = datasets.ImageFolder(root=f"{data_dir}/test", transform=image_transforms['test'])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

class_names = train_dataset.classes
num_classes = len(class_names)
print(f"Classes: {class_names} | Num classes: {num_classes}")

Classes: ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia'] | Num classes: 4


In [4]:
def get_class_counts(dataset):
    targets = dataset.targets
    class_to_idx = dataset.class_to_idx
    idx_to_class = {v: k for k, v in class_to_idx.items()}

    counts = {}
    for idx in idx_to_class:
        counts[idx] = targets.count(idx)
    return counts

class_counts = get_class_counts(train_dataset)
classes = list(class_counts.keys())
counts = list(class_counts.values())

total_samples = sum(counts)
weights = []
for i in range(num_classes):
    weight = total_samples / (num_classes * counts[i])
    weights.append(weight)

class_weights_tensor = torch.FloatTensor(weights).to(device)

print("Class counts:", class_counts)
print("Class weights:", weights)

Class counts: {0: 2892, 1: 4809, 2: 8153, 3: 1076}
Class weights: [1.4635200553250345, 0.880120607194843, 0.5191340610818104, 3.933550185873606]


In [5]:
# Number of isolated clients (hospitals)
num_clients = 5
dataset_size = len(train_dataset)

# Calculate split sizes
split_size = dataset_size // num_clients
lengths = [split_size] * num_clients
lengths[-1] += dataset_size % num_clients  # Add any remainder to the last client

# Split dataset randomly
client_datasets = random_split(
    train_dataset, 
    lengths, 
    generator=torch.Generator().manual_seed(42) # Seed ensures reproducibility
)

# Create 3 separate DataLoaders
client_trainloaders = []
for ds in client_datasets:
    client_trainloaders.append(DataLoader(ds, batch_size=32, shuffle=True, num_workers=2))

print(f"✅ Dataset split into {num_clients} clients with sizes: {[len(ds) for ds in client_datasets]}")

✅ Dataset split into 5 clients with sizes: [3386, 3386, 3386, 3386, 3386]


In [6]:
# ==========================================
# 1. LOAD & FREEZE TEACHER (DenseNet-121)
# ==========================================
print("Loading Teacher: DenseNet-121...")
teacher_model = models.densenet121(weights=None)
teacher_model.classifier = nn.Linear(1024, num_classes)

# Make sure this points to your exact DenseNet baseline weights
teacher_path = "/kaggle/input/datasets/rishwanthj/densenet121/best_densenet121.pth"
teacher_model.load_state_dict(torch.load(teacher_path, map_location=device))
teacher_model = teacher_model.to(device)
teacher_model.eval()

for p in teacher_model.parameters():
    p.requires_grad = False

# ==========================================
# 2. DEFINE STUDENT (HeteroMicroCNN)
# ==========================================
class HeteroMicroCNN(nn.Module):
    def __init__(self, num_classes):
        super(HeteroMicroCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(16), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            
            nn.Conv2d(16, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            
            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            
            nn.AdaptiveAvgPool2d((1, 1)) 
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3), 
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

print("✅ Architectures ready.")

Loading Teacher: DenseNet-121...
✅ Architectures ready.


In [7]:
# Hyperparameters for Distillation
KD_ALPHA = 0.7
KD_TEMP = 4.0

class HospitalClient(fl.client.NumPyClient):
    def __init__(self, student_model, teacher_model, train_loader, val_loader):
        self.student = student_model
        self.teacher = teacher_model
        self.train_loader = train_loader
        self.val_loader = val_loader

    def get_parameters(self, config):
        return [val.cpu().numpy() for _, val in self.student.state_dict().items()]

    def set_parameters(self, parameters):
        params_dict = zip(self.student.state_dict().keys(), parameters)
        state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
        self.student.load_state_dict(state_dict, strict=True)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        
        optimizer = optim.Adam(self.student.parameters(), lr=1e-4)
        ce_loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)
        kl_loss_fn = nn.KLDivLoss(reduction="none")
        
        self.student.train()
        
        # --- NEW: Train for 3 Local Epochs instead of 1 ---
        LOCAL_EPOCHS = 10
        
        for epoch in range(LOCAL_EPOCHS):
            for inputs, labels in self.train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.no_grad():
                    teacher_logits = self.teacher(inputs)
                    
                outputs = self.student(inputs)
                
                # Weighted KD Loss Calculation
                loss_ce = ce_loss_fn(outputs, labels)
                soft_student = F.log_softmax(outputs / KD_TEMP, dim=1)
                soft_teacher = F.softmax(teacher_logits / KD_TEMP, dim=1)
                
                sample_kd_loss = kl_loss_fn(soft_student, soft_teacher).sum(dim=1) 
                batch_weights = class_weights_tensor[labels]
                weighted_kd_loss = (sample_kd_loss * batch_weights).mean() * (KD_TEMP ** 2)
                
                loss = (1.0 - KD_ALPHA) * loss_ce + KD_ALPHA * weighted_kd_loss
                loss.backward()
                optimizer.step()

        return self.get_parameters(config={}), len(self.train_loader.dataset), {}
    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        self.student.eval()
        
        correct, total, loss = 0, 0, 0.0
        ce_loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)
        
        with torch.no_grad():
            for inputs, labels in self.val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = self.student(inputs)
                
                batch_loss = ce_loss_fn(outputs, labels)
                loss += batch_loss.item() * inputs.size(0)
                
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
                
        return loss / total, total, {"accuracy": correct / total}

In [8]:
# Function to generate a client dynamically during simulation
def client_fn(cid: str) -> fl.client.Client:
    client_id = int(cid)
    local_student = HeteroMicroCNN(num_classes).to(device)
    local_trainloader = client_trainloaders[client_id]
    return HospitalClient(local_student, teacher_model, local_trainloader, val_loader)

# --- NEW: Function to average the accuracy from the 3 hospitals ---
def weighted_average(metrics):
    # Multiply accuracy of each client by number of examples used
    accuracies = [num_examples * m["accuracy"] for num_examples, m in metrics]
    examples = [num_examples for num_examples, _ in metrics]
    # Return the aggregated accuracy
    return {"accuracy": sum(accuracies) / sum(examples)}
# ------------------------------------------------------------------

# Define the Aggregation Strategy
strategy = fl.server.strategy.FedAvg(
    fraction_fit=1.0,         
    fraction_evaluate=1.0,    
    min_fit_clients=3,        
    min_evaluate_clients=3,   
    min_available_clients=3,
    evaluate_metrics_aggregation_fn=weighted_average, # <-- NEW: Added this line
)

print("🚀 Starting Federated Distillation Simulation (DenseNet Pair)...")

# Run the Simulation
fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=3,
    config=fl.server.ServerConfig(num_rounds=35), 
    strategy=strategy,
    client_resources={"num_cpus": 2, "num_gpus": 1.0 if torch.cuda.is_available() else 0.0},
)

🚀 Starting Federated Distillation Simulation (DenseNet Pair)...


2026-03-01 08:40:48.608443: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772354448.766783      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772354448.822418      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772354449.229244      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772354449.229277      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772354449.229279      24 computation_placer.cc:177] computation placer alr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
INFO :      Starting Flower simulation, config: num_rounds=35, no round_timeout
2026-03-01 08:41:10,206	INFO worker.py:2012 -- Started a local Ray instance.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprec

History (loss, distributed):
	round 1: 1.0735176708998406
	round 2: 0.8322351466844532
	round 3: 0.7274414252212941
	round 4: 0.6511811161592953
	round 5: 0.5970761810601816
	round 6: 0.4878247047544725
	round 7: 0.5992791231075129
	round 8: 0.4599936921947405
	round 9: 0.41484289090244225
	round 10: 0.3998234482531614
	round 11: 0.399757693559393
	round 12: 0.44548029065114064
	round 13: 0.39059651800889095
	round 14: 0.3512430921404311
	round 15: 0.3466836822618253
	round 16: 0.3582999519354137
	round 17: 0.3689890587613018
	round 18: 0.34808918997693816
	round 19: 0.33169321351964215
	round 20: 0.3186359513604801
	round 21: 0.3243125768369527
	round 22: 0.3192459259670031
	round 23: 0.3481539984792017
	round 24: 0.3069173975609232
	round 25: 0.31358844133373776
	round 26: 0.28268418794274375
	round 27: 0.29084450758888664
	round 28: 0.3262792625166963
	round 29: 0.31306551986289816
	round 30: 0.36217531360158217
	round 31: 0.3014465535960285
	round 32: 0.3042671766499836
	round 33: 

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
